ETL Broadcast Publikasi Media Humas

Tempel teks broadcast mingguan di Langkah 3, lalu jalankan sel dari atas ke bawah. Berita yang belum ada akan masuk ke tabel `db_humas.publikasi_media_humas` di ClickHouse lokal.

Sebelum dipakai, pastikan ClickHouse lokal sudah jalan (`clickhousectl local server start`).

In [ ]:
%pip install clickhouse_connect

Note: you may need to restart the kernel to use updated packages.


1. Kode parser dan fungsi ETL

In [ ]:
import re
from datetime import date
from urllib.parse import parse_qsl, urlencode, urlparse, urlsplit
import pandas as pd
from clickhouse_connect import get_client

TABLE_NAME = "dataset.publikasi_media"
INVISIBLE = re.compile(r"[\u200b-\u200f\u2060\ufeff]")
OUTPUT_COLUMNS = [
    "ID", "JENIS BERITA", "TANGGAL AWAL", "TANGGAL AKHIR", "NAMA MEDIA",
    "SATKER_CODE", "JUDUL BERITA", "TAUTAN BERITA", "RAW", "LINK MEDIA",
]

SATKER_RULES = [
    ("POLITEKNIK STIA MKS", [r"politeknik\s+(?:stia\s+(?:lan\s+)?|sekolah\s+tinggi\s+ilmu\s+administrasi\s+lembaga\s+administrasi\s+negara\s+)(?:makassar|mks)\b", r"poltek\s+stia\s+(?:lan\s+)?(?:makassar|mks)\b", r"stia\s*lan\s+(?:makassar|mks)\b", r"stia\s*makassar"]),
    ("POLITEKNIK STIA JKT", [r"politeknik\s+(?:stia\s+(?:lan\s+)?|sekolah\s+tinggi\s+ilmu\s+administrasi\s+lembaga\s+administrasi\s+negara\s+)(?:jakarta|jkt)\b", r"poltek\s+stia\s+(?:lan\s+)?(?:jakarta|jkt)\b", r"stia\s*lan\s+(?:jakarta|jkt)\b", r"stia\s*jakarta"]),
    ("POLITEKNIK STIA BDG", [r"politeknik\s+(?:stia\s+(?:lan\s+)?|sekolah\s+tinggi\s+ilmu\s+administrasi\s+lembaga\s+administrasi\s+negara\s+)(?:bandung|bdg)\b", r"poltek\s+stia\s+(?:lan\s+)?(?:bandung|bdg)\b", r"stia\s*lan\s+(?:bandung|bdg)\b", r"stia\s*bandung"]),
    ("ACH", [r"pusjar\s*skmk", r"pusat\s+pembelajaran\s+standar\s+kompetensi\s+manajerial\s+dan\s+kepemimpinan", r"puslatbang\s*khan", r"pusat\s+pelatihan\s+dan\s+pengembangan\s+dan\s+kajian\s+hukum\s+administrasi\s+negara", r"\bkhan\b", r"\baceh\b"]),
    ("SMD", [r"pusjar\s*skpp", r"pusat\s+pembelajaran\s+standar\s+kompetensi\s+pelayanan\s+publik", r"puslatbang\s*kdod", r"pusat\s+pelatihan\s+dan\s+pengembangan\s+dan\s+kajian\s+desentralisasi\s+dan\s+otonomi\s+daerah", r"\bkdod\b", r"\bsamarinda\b"]),
    ("MKS", [r"pusjar\s*skmp", r"pusat\s+pembelajaran\s+standar\s+kompetensi\s+manajemen\s+pemerintahan", r"puslatbang\s*kmp", r"pusat\s+pelatihan\s+dan\s+pengembangan\s+dan\s+kajian\s+manajemen\s+pemerintahan", r"\bkmp\b", r"\bmakassar\b"]),
    ("BDG", [r"pusjar\s*sktan", r"pusjar\s*sk\s*tasnas", r"pusjar\s+strategi\s+kebijakan\s+talenta\s+asn\s+nasional", r"pusat\s+pembelajaran\s+standar\s+kompetensi\s+dan\s+talenta\s+asn\s+nasional", r"pusat\s+pembelajaran\s+dan\s+strategi\s+kebijakan\s+talenta\s+asn\s+nasional", r"puslatbang\s*pkasn", r"pusat\s+pelatihan\s+dan\s+pengembangan\s+dan\s+pemetaan\s+kompetensi\s+dan\s+penilaian\s+kinerja\s+asn", r"\bpkasn\b", r"\bjatinangor\b", r"\bbandung\b"]),
]

def get_satker(text: str):
    t = str(text).lower()
    for code, patterns in SATKER_RULES:
        if any(re.search(p, t) for p in patterns):
            return code
    return None

def get_section_header(text: str):
    label = re.sub(r"[*_`]+", "", text).strip().rstrip(":").strip()
    patterns = [
        (r"media\s+sosial", "MEDIA SOSIAL"),
        (r"(?:youtube|instagram|twitter(?:\s*\(x\))?|tiktok|facebook)(?:\s*[-:].*)?", "MEDIA SOSIAL"),
        (r"media\s+(?:massa|online)", "MEDIA MASSA"),
        (r"(?:website(?:\s+lan)?|berita\s+terkini|awalan(?:\s*\([^)]*\))?)", "BERITA TERKINI"),
        (r"berita\s*unit", "BERITA UNIT"),
    ]
    for pattern, section in patterns:
        if re.fullmatch(pattern, label, re.I):
            return section
    return None

MONTHS_ID = {
    "januari": 1, "februari": 2, "pebruari": 2, "maret": 3, "april": 4,
    "mei": 5, "juni": 6, "juli": 7, "agustus": 8, "september": 9,
    "oktober": 10, "november": 11, "desember": 12,
}

def parse_report_range(text):
    text = text.strip().strip("*_ ")
    compact = re.fullmatch(r"(?P<start>\d{1,2})\s*[-–]\s*(?P<end>\d{1,2})\s+(?P<month>[\w]+)\s+(?P<year>\d{4})", text, re.I)
    cross_month = re.fullmatch(
        r"(?P<start>\d{1,2})\s+(?P<start_month>[\w]+)\s*[-–]\s*(?P<end>\d{1,2})\s+(?P<end_month>[\w]+)\s+(?P<year>\d{4})",
        text,
        re.I,
    )
    full_dates = re.fullmatch(
        r"(?:[\w’']+\s+)?(?P<start>\d{1,2})\s+(?P<start_month>[\w]+)\s+(?P<start_year>\d{4})"
        r"\s*[-–]\s*(?:[\w’']+\s+)?(?P<end>\d{1,2})\s+(?P<end_month>[\w]+)\s+(?P<end_year>\d{4})",
        text,
        re.I,
    )
    try:
        if compact:
            month = MONTHS_ID.get(compact.group("month").lower())
            if month:
                year = int(compact.group("year"))
                return date(year, month, int(compact.group("start"))), date(year, month, int(compact.group("end")))
        if cross_month:
            start_month = MONTHS_ID.get(cross_month.group("start_month").lower())
            end_month = MONTHS_ID.get(cross_month.group("end_month").lower())
            if start_month and end_month:
                year = int(cross_month.group("year"))
                return date(year, start_month, int(cross_month.group("start"))), date(
                    year, end_month, int(cross_month.group("end"))
                )
        if full_dates:
            start_month = MONTHS_ID.get(full_dates.group("start_month").lower())
            end_month = MONTHS_ID.get(full_dates.group("end_month").lower())
            if start_month and end_month:
                return date(int(full_dates.group("start_year")), start_month, int(full_dates.group("start"))), date(
                    int(full_dates.group("end_year")), end_month, int(full_dates.group("end"))
                )
    except ValueError:
        return None
    return None

def get_media_name(url: str):
    domain = (urlparse(url).hostname or "").lower().removeprefix("www.")
    if "youtube.com" in domain or "youtu.be" in domain: return "YOUTUBE"
    if "instagram.com" in domain: return "INSTAGRAM"
    if "tiktok.com" in domain: return "TIKTOK"
    if "twitter.com" in domain or "x.com" in domain: return "TWITTER (X)"
    if "facebook.com" in domain: return "FACEBOOK"
    if "lan.go.id" in domain: return "LAN RI"
    if domain == "sindonews.com" or domain.endswith(".sindonews.com"): return "SINDONEWS"
    if domain == "antaranews.com" or domain.endswith(".antaranews.com"): return "ANTARANEWS"
    if domain == "tribunnews.com" or domain.endswith(".tribunnews.com"): return "TRIBUNNEWS"
    return domain.split(".")[0].upper() if domain else None

def clean_url(url: str) -> str:
    url = str(url).strip()
    schemes = list(re.finditer(r"https?://", url, re.I))
    if len(schemes) > 1:
        url = url[schemes[-1].start():]
    return url

def url_key(url: str) -> str:
    parsed = urlsplit(clean_url(url))
    query = [
        (key, value) for key, value in parse_qsl(parsed.query)
        if not key.lower().startswith("utm_") and key.lower() not in {"fbclid", "igshid"}
    ]
    host = parsed.netloc.lower().removeprefix("www.")
    path = re.sub(r"^/amp(?=/)|/amp$", "", parsed.path.rstrip("/"), flags=re.I)
    return f"{host}{path}?{urlencode(query)}".rstrip("?")

def append_new_records(df_records, client):
    columns = [
        "id", "jenis_berita", "tanggal_awal", "tanggal_akhir", "nama_media",
        "satker_code", "judul_berita", "tautan_berita", "raw", "link_media",
    ]
    existing_rows = client.query(f"SELECT id, tautan_berita FROM {TABLE_NAME}").result_rows
    existing_keys = {url_key(row[1]) for row in existing_rows if row[1]}
    next_id = max((int(row[0]) for row in existing_rows), default=0) + 1

    df_new = df_records.copy()
    df_new["_key"] = df_new["TAUTAN BERITA"].map(url_key)
    df_new = df_new.drop_duplicates(subset=["_key"], keep="first")
    df_new = df_new[~df_new["_key"].isin(existing_keys)].drop(columns="_key").copy()
    if df_new.empty:
        df_new["ID"] = pd.Series(dtype="int64")
        return df_new.reindex(columns=OUTPUT_COLUMNS)

    df_new.insert(0, "ID", range(next_id, next_id + len(df_new)))

    def nullable_text(value):
        return None if pd.isna(value) else str(value)

    def nullable_date(value):
        return None if pd.isna(value) else pd.to_datetime(value).date()

    payload = [[
        int(row["ID"]), str(row["JENIS BERITA"]), nullable_date(row["TANGGAL AWAL"]),
        nullable_date(row["TANGGAL AKHIR"]), nullable_text(row["NAMA MEDIA"]),
        nullable_text(row["SATKER_CODE"]), str(row["JUDUL BERITA"] or ""),
        str(row["TAUTAN BERITA"]), nullable_text(row["RAW"]), nullable_text(row["LINK MEDIA"]),
    ] for _, row in df_new.iterrows()]
    client.insert(TABLE_NAME, payload, column_names=columns)
    return df_new.reindex(columns=OUTPUT_COLUMNS)

def parse_chat_mingguan(raw_chat: str) -> pd.DataFrame:
    """Memecah teks chat broadcast WA langsung menjadi DataFrame siap pakai."""
    records, current_sec, buf_title = [], None, ""
    report_start = report_end = None
    message_start = re.compile(r"^\d{1,2}/\d{1,2}/\d{2,4},?\s+\d{1,2}[.:]\d{2}\s+-\s+[^:]+:\s*(.*)$")
    item_date = re.compile(r"\d{1,2}\s*/\s*\d{1,2}(?:\s*/\s*\d{2,4})?")
    markdown_link = re.compile(r"\[([^\]]*)\]\((https?://[^)\s]+)\)")
    url_pattern = re.compile(r"https?://[^\s<>\]]+", re.I)
    social_tail = re.compile(r"\*(?=(?:youtube|instagram|twitter|tiktok|facebook)\b)", re.I)
    noise = re.compile(r"<media\s+(?:tidak\s+)?disertakan(?:\s+tidak)?>|program\s+germas|cek\s+kesehatan|berita\s+duka\s+cita|telah\s+(?:berpulang|meninggal\s+dunia)", re.I)

    for raw_line in raw_chat.splitlines():
        raw_line = INVISIBLE.sub("", raw_line)
        line = raw_line.strip()
        if not line:
            continue

        message = message_start.match(line)
        if message:
            current_sec, buf_title = None, ""
            report_start = report_end = None
            line = message.group(1).strip()
            if not line:
                continue

        date_range = parse_report_range(line)
        if date_range:
            report_start, report_end = date_range
            buf_title = ""
            continue

        if item_date.fullmatch(line):
            buf_title = ""
            continue

        if noise.search(line):
            buf_title = ""
            continue

        header = get_section_header(line)
        if header:
            current_sec, buf_title = header, ""
            continue

        if current_sec is None:
            continue

        line = markdown_link.sub(
            lambda match: (match.group(1) + " " if not match.group(1).strip().lower().startswith("http") else "") + match.group(2),
            line,
        )
        url_match = url_pattern.search(line)
        if not url_match:
            buf_title = f"{buf_title} {line}".strip()
            continue

        url = clean_url(url_match.group(0))
        trailing = line[url_match.end():]
        glued_header = social_tail.search(url)
        if glued_header:
            trailing = url[glued_header.start():] + trailing
            url = url[:glued_header.start()]
        url = url.rstrip(".,;:!?)]}>")
        inline_title = line[:url_match.start()]
        title = re.sub(r"^\s*(?:\*?\d+[.)]\s*|[-•]\s*)", "", f"{buf_title} {inline_title}")
        title = re.sub(r"[*_`]+", "", title).strip(" \t:-")
        media_name = get_media_name(url)
        link_media = urlparse(url).netloc.lower().removeprefix("www.") or None

        records.append({
            "JENIS BERITA": current_sec,
            "TANGGAL AWAL": report_start,
            "TANGGAL AKHIR": report_end,
            "JUDUL BERITA": title if title else "Publikasi Media LAN",
            "TAUTAN BERITA": url,
            "NAMA MEDIA": media_name,
            "SATKER_CODE": get_satker(title) if current_sec == "BERITA UNIT" else None,
            "RAW": f"{buf_title} {raw_line.strip()}".strip(),
            "LINK MEDIA": link_media,
        })
        buf_title = ""

        trailing_header = get_section_header(trailing)
        if trailing_header:
            current_sec = trailing_header

    record_columns = [
        "JENIS BERITA", "TANGGAL AWAL", "TANGGAL AKHIR", "NAMA MEDIA",
        "SATKER_CODE", "JUDUL BERITA", "TAUTAN BERITA", "RAW", "LINK MEDIA",
    ]
    return pd.DataFrame(records, columns=record_columns)


2. Koneksi ke ClickHouse

In [ ]:
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

client = get_client(host=, port=, username=, password=, database=)
print("Terhubung ke ClickHouse. Tabel:", TABLE_NAME)

Terhubung ke ClickHouse. Tabel: dataset.publikasi_media


3. Tempel teks broadcast
Ganti isi di antara tanda `"""` dengan teks broadcast lengkap.

In [ ]:
BROADCAST = r"""
Ada Apa di Lembaga Administrasi Negara
9-15 November 2026

Media Online:
1. BPS Anugerahi Tiga Lembaga Pemerintah Penghargaan Statistik Sektoral https://www.cnnindonesia.com/nasional/20261112131723-293-1426343/bps-anugerahi-tiga-lembaga-pemerintah-penghargaan-statistik-sektoral
2. Tingkatkan Kualitas Birokrasi, Pemkab Bartim Matangkan Sistem Pembelajaran ASN Terintegrasi https://lenterakalimantan.com/2026/07/tingkatkan-kualitas-birokrasi-pemkab-bartim-matangkan-sistem-pembelajaran-asn-terintegrasi/
3. Bupati Muaro Jambi Hadiri Penutupan PKN II di LAN RI, Dorong ASN Terapkan Inovasi untuk Daerah https://jambiindependent.disway.id/amp/720443/bupati-muaro-jambi-hadiri-penutupan-pkn-ii-di-lan-ri-dorong-asn-terapkan-inovasi-untuk-daerah
4. Sekwan DPRD Medan Gandeng LAN RI, Wujudkan Akselerasi Pelayanan Lewat Laboratorium Inovasi https://www.agiodeli.id/2026/11/sekwan-dprd-medan-gandeng-lan-ri.html
5. 8 Kepala OPD Pemkab Minut Lulus PKN II Angkatan XXIII, LAN RI Ingatkan Pentingnya Kontribusi Nyata https://petasulut.com/8-kepala-opd-pemkab-minut-lulus-pkn-ii-angkatan-xxiii-lan-ri-ingatkan-pentingnya-kontribusi-nyata/
6. Pemkot Medan & LAN RI Teken Kerja Sama Tingkatkan Kapasitas ASN https://inspirasinews.id/2026/11/pemkot-medan-lan-ri-teken-kerja-sama/
7. Talenta Nusantara Rancang Penugasan ASN Lintas Instansi untuk Misi Strategis Nasional https://aceh.tribunnews.com/nanggroe/1054056/talenta-nusantara-rancang-penugasan-asn-lintas-instansi-untuk-misi-strategis-nasional
8. Teken Nota Kesepakatan dengan LAN RI, Rico Pacu Transformasi Birokrasi dan Inovasi https://rri.co.id/medan/iptek/2883151/teken-nota-kesepakatan-dengan-lan-ri-rico-pacu-transformasi-birokrasi-dan-inovasi
9. Pemko Medan bersama LANRI perkuat inovasi pelayanan public https://suarasumutonline.id/medan/pemko-medan-bersama-lan-ri-perkuat-inovasi-pelayanan-publik/
10. LAN RI Tantang Pemko Medan Lahirkan Inovasi yang Dirasakan Warga https://medan.tribunnews.com/adv/1818974/lan-ri-tantang-pemko-medan-lahirkan-inovasi-yang-dirasakan-warga
11. Akselerasi Layanan Laboratorium Inovasi Pemko Medan dan LAN RI, Rico: Harus Berdampak ke Masyarakat https://medan.tribunnews.com/adv/1818964/akselerasi-layanan-laboratorium-inovasi-pemko-medan-dan-lan-ri-rico-harus-berdampak-ke-masyarakat
12. Teken Nota Kesepakatan dengan LAN RI, Rico Waas Pacu Transformasi Birokrasi dan Inovasi Pelayanan Publik https://asarpua.com/teken-nota-kesepakatan-dengan-lan-ri-rico-waas-pacu-transformasi-birokrasi-dan-inovasi-pelayanan-publik/
13. Dari Riset Menuju Kebijakan, Politeknik STIA LAN Jakarta Dorong Transformasi Manajemen Talenta ASN Nasional Berbasis AI https://indonews.id/artikel/356096/Dari-Riset-Menuju-Kebijakan-Politeknik-STIA-LAN-Jakarta-Dorong-Transformasi-Manajemen-Talenta-ASN-Nasional-Berbasis-AI/
14. Politeknik STIA LAN Jakarta Ambil Peran sebagai Petugas Upacara Hari Pahlawan 2026 https://indonews.id/artikel/356045/Politeknik-STIA-LAN-Jakarta-Ambil-Peran-sebagai-Petugas-Upacara-Hari-Pahlawan-2026/
15. Perkuat Statistik Sektoral, BRIN, LKPP dan LAN Raih Penghargaan BPS https://rm.id/baca-berita/government-action/334246/perkuat-statistik-sektoral-brin-lkpp-dan-lan-raih-penghargaan-bps
16. Laboratorium Inovasi LAN RI Diharapkan Buat Pelayanan Lebih Cepat https://inspirasinews.id/2026/11/laboratorium-inovasi-lan-ri-diharapkan-pelayanan-lebih-cepat/
17. Sekretariat DPRD Medan dan Perangkat Daerah Ikuti Laboratorium Inovasi dengan LAN RI https://asarpua.com/sekretariat-dprd-medan-dan-perangkat-daerah-ikuti-laboratorium-inovasi-dengan-lan-ri/
18. Tauwwa, Hadirkan Pemateri LAN RI, FKP Malut Makassar Gelar Sekolah Policy Brief https://bugispos.com/tauwwa-hadirkan-pemateri-lan-ri-fkp-malut-makassar-gelar-sekolah-policy-brief/
19. Transformasi Pembelajaran Kepemimpinan Untuk Pemimpin Birokrasi Masa Depan https://www.kompasiana.com/ditjarmapim16173/6af53e7a34777c31f97c3287/transformasi-pembelajaran-kepemimpinan-demi-pemimpin-birokrasi-masa-depan#goog_rewarded
20. ⁠GANESA 2026: Menebar Benih Literasi dan Inklusi di Microlibrary Asia Afrika https://jabar.tribunnews.com/kampus-360/1189927/ganesa-2026-menebar-benih-literasi-dan-inklusi-di-microlibrary-asia-afrika#goog_rewarded
"""

4. Parsing dan pratinjau

In [ ]:
df = parse_chat_mingguan(BROADCAST)
print(f"Ditemukan {len(df)} berita")
if df.empty:
    print("Tidak ada berita terbaca. Cek penulisan periode tanggal dan header seksi (Media Online, Berita Unit, dst).")
else:
    print(df["JENIS BERITA"].value_counts().to_string())
    print("Periode:", df["TANGGAL AWAL"].iloc[0], "s.d.", df["TANGGAL AKHIR"].iloc[0])
df[["JENIS BERITA", "SATKER_CODE", "JUDUL BERITA", "TAUTAN BERITA"]]

Ditemukan 20 berita
JENIS BERITA
MEDIA MASSA    20
Periode: 2026-11-09 s.d. 2026-11-15


,JENIS BERITA,SATKER_CODE,JUDUL BERITA,TAUTAN BERITA
0,MEDIA MASSA,None,BPS Anugerahi Tiga Lembaga Pemerintah Penghargaan Statistik Sektoral,https://www.cnnindonesia.com/nasional/20261112131723-293-1426343/bps-anugerahi-tiga-le...
1,MEDIA MASSA,None,"Tingkatkan Kualitas Birokrasi, Pemkab Bartim Matangkan Sistem Pembelajaran ASN Terinte...",https://lenterakalimantan.com/2026/07/tingkatkan-kualitas-birokrasi-pemkab-bartim-mata...
2,MEDIA MASSA,None,"Bupati Muaro Jambi Hadiri Penutupan PKN II di LAN RI, Dorong ASN Terapkan Inovasi untu...",https://jambiindependent.disway.id/amp/720443/bupati-muaro-jambi-hadiri-penutupan-pkn-...
3,MEDIA MASSA,None,"Sekwan DPRD Medan Gandeng LAN RI, Wujudkan Akselerasi Pelayanan Lewat Laboratorium Ino...",https://www.agiodeli.id/2026/11/sekwan-dprd-medan-gandeng-lan-ri.html
4,MEDIA MASSA,None,"8 Kepala OPD Pemkab Minut Lulus PKN II Angkatan XXIII, LAN RI Ingatkan Pentingnya Kont...",https://petasulut.com/8-kepala-opd-pemkab-minut-lulus-pkn-ii-angkatan-xxiii-lan-ri-ing...
5,MEDIA MASSA,None,Pemkot Medan & LAN RI Teken Kerja Sama Tingkatkan Kapasitas ASN,https://inspirasinews.id/2026/11/pemkot-medan-lan-ri-teken-kerja-sama/
6,MEDIA MASSA,None,Talenta Nusantara Rancang Penugasan ASN Lintas Instansi untuk Misi Strategis Nasional,https://aceh.tribunnews.com/nanggroe/1054056/talenta-nusantara-rancang-penugasan-asn-l...
7,MEDIA MASSA,None,"Teken Nota Kesepakatan dengan LAN RI, Rico Pacu Transformasi Birokrasi dan Inovasi",https://rri.co.id/medan/iptek/2883151/teken-nota-kesepakatan-dengan-lan-ri-rico-pacu-t...
8,MEDIA MASSA,None,Pemko Medan bersama LANRI perkuat inovasi pelayanan public,https://suarasumutonline.id/medan/pemko-medan-bersama-lan-ri-perkuat-inovasi-pelayanan...
9,MEDIA MASSA,None,LAN RI Tantang Pemko Medan Lahirkan Inovasi yang Dirasakan Warga,https://medan.tribunnews.com/adv/1818974/lan-ri-tantang-pemko-medan-lahirkan-inovasi-y...


5. Cek berita yang belum ada di ClickHouse

In [ ]:
rows = client.query(f"SELECT tautan_berita FROM {TABLE_NAME}").result_rows
sudah_ada = {url_key(r[0]) for r in rows if r[0]}
cek = df.copy()
cek["_key"] = cek["TAUTAN BERITA"].map(url_key)
cek = cek.drop_duplicates("_key")
baru = cek[~cek["_key"].isin(sudah_ada)]
print(f"Dari {len(df)} berita: {len(baru)} baru, {len(cek) - len(baru)} sudah ada atau dobel.")
baru[["JENIS BERITA", "SATKER_CODE", "JUDUL BERITA", "TAUTAN BERITA"]]

Dari 20 berita: 20 baru, 0 sudah ada atau dobel.


,JENIS BERITA,SATKER_CODE,JUDUL BERITA,TAUTAN BERITA
0,MEDIA MASSA,None,BPS Anugerahi Tiga Lembaga Pemerintah Penghargaan Statistik Sektoral,https://www.cnnindonesia.com/nasional/20261112131723-293-1426343/bps-anugerahi-tiga-le...
1,MEDIA MASSA,None,"Tingkatkan Kualitas Birokrasi, Pemkab Bartim Matangkan Sistem Pembelajaran ASN Terinte...",https://lenterakalimantan.com/2026/07/tingkatkan-kualitas-birokrasi-pemkab-bartim-mata...
2,MEDIA MASSA,None,"Bupati Muaro Jambi Hadiri Penutupan PKN II di LAN RI, Dorong ASN Terapkan Inovasi untu...",https://jambiindependent.disway.id/amp/720443/bupati-muaro-jambi-hadiri-penutupan-pkn-...
3,MEDIA MASSA,None,"Sekwan DPRD Medan Gandeng LAN RI, Wujudkan Akselerasi Pelayanan Lewat Laboratorium Ino...",https://www.agiodeli.id/2026/11/sekwan-dprd-medan-gandeng-lan-ri.html
4,MEDIA MASSA,None,"8 Kepala OPD Pemkab Minut Lulus PKN II Angkatan XXIII, LAN RI Ingatkan Pentingnya Kont...",https://petasulut.com/8-kepala-opd-pemkab-minut-lulus-pkn-ii-angkatan-xxiii-lan-ri-ing...
5,MEDIA MASSA,None,Pemkot Medan & LAN RI Teken Kerja Sama Tingkatkan Kapasitas ASN,https://inspirasinews.id/2026/11/pemkot-medan-lan-ri-teken-kerja-sama/
6,MEDIA MASSA,None,Talenta Nusantara Rancang Penugasan ASN Lintas Instansi untuk Misi Strategis Nasional,https://aceh.tribunnews.com/nanggroe/1054056/talenta-nusantara-rancang-penugasan-asn-l...
7,MEDIA MASSA,None,"Teken Nota Kesepakatan dengan LAN RI, Rico Pacu Transformasi Birokrasi dan Inovasi",https://rri.co.id/medan/iptek/2883151/teken-nota-kesepakatan-dengan-lan-ri-rico-pacu-t...
8,MEDIA MASSA,None,Pemko Medan bersama LANRI perkuat inovasi pelayanan public,https://suarasumutonline.id/medan/pemko-medan-bersama-lan-ri-perkuat-inovasi-pelayanan...
9,MEDIA MASSA,None,LAN RI Tantang Pemko Medan Lahirkan Inovasi yang Dirasakan Warga,https://medan.tribunnews.com/adv/1818974/lan-ri-tantang-pemko-medan-lahirkan-inovasi-y...


6. Jalankan ETL (simpan ke ClickHouse)

In [ ]:
hasil = append_new_records(df, client)
print(f"Berhasil menambahkan {len(hasil)} berita baru ke {TABLE_NAME}.")
if hasil.empty:
    print("Tidak ada URL baru; seluruh tautan sudah ada di ClickHouse.")
hasil[["ID", "JENIS BERITA", "SATKER_CODE", "JUDUL BERITA", "TAUTAN BERITA"]]

Berhasil menambahkan 20 berita baru ke dataset.publikasi_media.


,ID,JENIS BERITA,SATKER_CODE,JUDUL BERITA,TAUTAN BERITA
0,7827,MEDIA MASSA,None,BPS Anugerahi Tiga Lembaga Pemerintah Penghargaan Statistik Sektoral,https://www.cnnindonesia.com/nasional/20261112131723-293-1426343/bps-anugerahi-tiga-le...
1,7828,MEDIA MASSA,None,"Tingkatkan Kualitas Birokrasi, Pemkab Bartim Matangkan Sistem Pembelajaran ASN Terinte...",https://lenterakalimantan.com/2026/07/tingkatkan-kualitas-birokrasi-pemkab-bartim-mata...
2,7829,MEDIA MASSA,None,"Bupati Muaro Jambi Hadiri Penutupan PKN II di LAN RI, Dorong ASN Terapkan Inovasi untu...",https://jambiindependent.disway.id/amp/720443/bupati-muaro-jambi-hadiri-penutupan-pkn-...
3,7830,MEDIA MASSA,None,"Sekwan DPRD Medan Gandeng LAN RI, Wujudkan Akselerasi Pelayanan Lewat Laboratorium Ino...",https://www.agiodeli.id/2026/11/sekwan-dprd-medan-gandeng-lan-ri.html
4,7831,MEDIA MASSA,None,"8 Kepala OPD Pemkab Minut Lulus PKN II Angkatan XXIII, LAN RI Ingatkan Pentingnya Kont...",https://petasulut.com/8-kepala-opd-pemkab-minut-lulus-pkn-ii-angkatan-xxiii-lan-ri-ing...
5,7832,MEDIA MASSA,None,Pemkot Medan & LAN RI Teken Kerja Sama Tingkatkan Kapasitas ASN,https://inspirasinews.id/2026/11/pemkot-medan-lan-ri-teken-kerja-sama/
6,7833,MEDIA MASSA,None,Talenta Nusantara Rancang Penugasan ASN Lintas Instansi untuk Misi Strategis Nasional,https://aceh.tribunnews.com/nanggroe/1054056/talenta-nusantara-rancang-penugasan-asn-l...
7,7834,MEDIA MASSA,None,"Teken Nota Kesepakatan dengan LAN RI, Rico Pacu Transformasi Birokrasi dan Inovasi",https://rri.co.id/medan/iptek/2883151/teken-nota-kesepakatan-dengan-lan-ri-rico-pacu-t...
8,7835,MEDIA MASSA,None,Pemko Medan bersama LANRI perkuat inovasi pelayanan public,https://suarasumutonline.id/medan/pemko-medan-bersama-lan-ri-perkuat-inovasi-pelayanan...
9,7836,MEDIA MASSA,None,LAN RI Tantang Pemko Medan Lahirkan Inovasi yang Dirasakan Warga,https://medan.tribunnews.com/adv/1818974/lan-ri-tantang-pemko-medan-lahirkan-inovasi-y...


7. Verifikasi di ClickHouse

In [ ]:
print(client.query(f"SELECT count(), max(id), uniqExact(id) FROM {TABLE_NAME} FINAL").result_rows)
client.query_df(f"""
    SELECT id, jenis_berita, tanggal_awal, tanggal_akhir, nama_media, satker_code, judul_berita
    FROM {TABLE_NAME} FINAL
    ORDER BY id DESC LIMIT 20
""")

[(7846, 7846, 7846)]


,id,jenis_berita,tanggal_awal,tanggal_akhir,nama_media,satker_code,judul_berita
0,7846,MEDIA MASSA,2026-11-09,2026-11-15,TRIBUNNEWS,<NA>,GANESA 2026: Menebar Benih Literasi dan Inklusi di Microlibrary Asia Afrika
1,7845,MEDIA MASSA,2026-11-09,2026-11-15,KOMPASIANA,<NA>,Transformasi Pembelajaran Kepemimpinan Untuk Pemimpin Birokrasi Masa Depan
2,7844,MEDIA MASSA,2026-11-09,2026-11-15,BUGISPOS,<NA>,"Tauwwa, Hadirkan Pemateri LAN RI, FKP Malut Makassar Gelar Sekolah Policy Brief"
3,7843,MEDIA MASSA,2026-11-09,2026-11-15,ASARPUA,<NA>,Sekretariat DPRD Medan dan Perangkat Daerah Ikuti Laboratorium Inovasi dengan LAN RI
4,7842,MEDIA MASSA,2026-11-09,2026-11-15,INSPIRASINEWS,<NA>,Laboratorium Inovasi LAN RI Diharapkan Buat Pelayanan Lebih Cepat
5,7841,MEDIA MASSA,2026-11-09,2026-11-15,RM,<NA>,"Perkuat Statistik Sektoral, BRIN, LKPP dan LAN Raih Penghargaan BPS"
6,7840,MEDIA MASSA,2026-11-09,2026-11-15,INDONEWS,<NA>,Politeknik STIA LAN Jakarta Ambil Peran sebagai Petugas Upacara Hari Pahlawan 2026
7,7839,MEDIA MASSA,2026-11-09,2026-11-15,INDONEWS,<NA>,"Dari Riset Menuju Kebijakan, Politeknik STIA LAN Jakarta Dorong Transformasi Manajemen..."
8,7838,MEDIA MASSA,2026-11-09,2026-11-15,ASARPUA,<NA>,"Teken Nota Kesepakatan dengan LAN RI, Rico Waas Pacu Transformasi Birokrasi dan Inovas..."
9,7837,MEDIA MASSA,2026-11-09,2026-11-15,TRIBUNNEWS,<NA>,"Akselerasi Layanan Laboratorium Inovasi Pemko Medan dan LAN RI, Rico: Harus Berdampak ..."
